# Data Exploration
This notebook explores the source datasets and records the key assumptions used for the transformations at each layer.



## Contents
1. Key configurations
2. Sampling the files (Raw → Bronze)
3. Exploring the tables (Bronze → Silver)
   - 3.1 Customers
   - 3.2 Products
   - 3.3 Orders

### 3.1 Customers: key findings
1. No duplicate `customer_id`s.
2. `customer_name` needs cleaning: remove special characters and collapse repeated spaces.
3. `phone` needs cleaning: remove letters and special characters, and format as a 10-digit US number.

### 3.2 Products: key findings
1. `product_id` is duplicated, so deduplicate before joining.
2. Category and sub-category can be derived from `product_id`, e.g. `FUR-CH` → Furniture, Chairs.
3. `price_per_product` has missing values.
4. A product classification lookup table will map the `product_id` shorthands to category and sub-category.

### 3.3 Orders: key findings
1. All `customer_id`s match the customer table.
2. Not all `product_id`s match the product table.
3. The product classification lookup fills in category and sub-category for the unmatched products.
4. `order_year` is derived from `order_date` and used for the aggregations.

# 1 Key Configurations

In [0]:
%pip install openpyxl

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import  StructType, StringType, StructField, IntegerType
import re, pandas as pd
import os

#Catalog & Schema
CATALOG= "DE_ASSESSMENT_DEV"
SCHEMA_BRONZE = "BRONZE" 
SCHEMA_SILVER = "SILVER" 
SCHEMA_GOLD = "GOLD" 

#Input file paths 
file_name_customer ="Customer.xlsx"
file_name_orders ="Orders.json"
file_name_product ="Products.csv"

#row counts manually extracted for validation
row_count_customer = 793
row_count_orders = 9994
row_count_products = 1851

#Raw data volume path
input_folder_path = os.path.join(os.getcwd(), "data")

#Column names of the input tables. Calculated using cell 4 output of 00_exploratory.ipynb . Used for schema validation
columns_names_orders = ['Customer ID','Discount', 'Order Date', 'Order ID', 'Price', 'Product ID', 'Profit', 'Quantity', 'Row ID', 'Ship Date', 'Ship Mode']
columns_names_products = ['Product ID', 'Category', 'Sub-Category', 'Product Name', 'State', 'Price per product']
columns_names_customer = ['Customer ID', 'Customer Name', 'email', 'phone', 'address', 'Segment', 'Country', 'City', 'State', 'Postal Code', 'Region']


In [0]:
# Create the catalog and schema
spark.sql (f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql (f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA_BRONZE}")
spark.sql (f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA_SILVER}")
spark.sql (f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA_GOLD}") 

#2  Sampling the files [Raw -> Bronze]

In [0]:
print(dbutils.fs.ls(f"{input_folder_path}"))
print(dbutils.fs.head(f"{input_folder_path}/{file_name_orders}",400))
print(dbutils.fs.head(f"{input_folder_path}/{file_name_product}",200))

In [0]:

customer_xl =pd.ExcelFile(f"{input_folder_path}/{file_name_customer}").sheet_names
print(customer_xl)


In [0]:
explore_orders = spark.read.option("multiLine", True).json(f"{input_folder_path}/{file_name_orders}")
explore_product = spark.read.option("header",True).csv(f"{input_folder_path}/{file_name_product}")
explore_customer = pd.read_excel(f"{input_folder_path}/{file_name_customer}",'Worksheet')


for table_name, columns in [
                            ("orders" , explore_orders.columns),
                            ("products" , explore_product.columns),
                            ("customers" , list(explore_customer.columns)),
                            ]:
    print(f"{table_name}_columns = {columns}")

In [0]:

raw_orders = (spark.read.option("multiLine", True)
                        .option("mode","PERMISSIVE")
                        .option("rescuedDataColumn","_rescued_data")
                        .option("primitivesAsString",True)
                        .json(f"{input_folder_path}/{file_name_orders}")
                )

display(raw_orders.limit(5))

raw_products = (spark.read.option("header",True)
                        .option("enforceSchema",False)
                        .option("rescuedDataColumn","_rescued_data")
                        .option("mode","PERMISSIVE")
                        .option("primitivesAsString",True)                        
                        .csv(f"{input_folder_path}/{file_name_product}")
                )

display(raw_products.limit(5))

raw_pdf_customer =pd.read_excel(f"{input_folder_path}/{file_name_customer}",'Worksheet',dtype ="str")

raw_pdf_customer = raw_pdf_customer.astype("object").where(raw_pdf_customer.notna(),None)
raw_customer = spark.createDataFrame(raw_pdf_customer)

display(raw_customer.limit(100))


In [0]:
import re
def rename(columns):
    return [re.sub(r"[^A-Za-z0-9]+","_",column).lower() for column in columns]

def to_bronze(df, table, source):
    final_df = (df.withColumn("_source",F.lit(source))
                   .withColumn("_ingested_at",F.current_timestamp())
    )

    (final_df.write
            .mode('overwrite')
            .option('overwriteSchema', True)
            .saveAsTable(table)
    )
    return spark.table(table)

In [0]:
def dq_check_schema(actual_columns , expected_columns):
    actual_columns = set(actual_columns) -set(["_rescued_data"])
    if set(actual_columns)  != set(expected_columns):
        raise Exception(f"Schema does not match expected schema. Actual: {actual_columns}, Expected: {expected_columns}")
    else:
        print("Schema valid")

def dq_check_rows(actual, expected):
    if actual != expected:
        raise Exception(f"Number of rows does not match. Actual: {actual}, Expected: {expected}")
    else:
        print("Number of rows valid")

In [0]:
dq_check_schema(raw_orders.columns,columns_names_orders)
dq_check_schema(raw_customer.columns,columns_names_customer)
dq_check_schema(raw_products.columns,columns_names_products)

dq_check_rows(raw_orders.count(),row_count_orders)
dq_check_rows(raw_customer.count(),row_count_customer)
dq_check_rows(raw_products.count(),row_count_products)

bronze_orders=raw_orders.toDF(*rename(raw_orders.columns))
bronze_customer = raw_customer.toDF(*rename(raw_customer.columns))
bronze_products = raw_products.toDF(*rename(raw_products.columns))

to_bronze(bronze_orders, f"{CATALOG}.{SCHEMA_BRONZE}.orders",f"{input_folder_path}/{file_name_orders}")
to_bronze(bronze_customer, f"{CATALOG}.{SCHEMA_BRONZE}.customer",f"{input_folder_path}/{file_name_customer}")
to_bronze(bronze_products, f"{CATALOG}.{SCHEMA_BRONZE}.products",f"{input_folder_path}/{file_name_product}")



# Part 3: Exploring Data with focus on  Data Cleaning & Validation
## Moving Data from Bronze to Silver


### 3.1 Exploring Customer Table
Key Findings
1. no duplicates customer_ids
2. Customer Name needs cleaning, replacing multiple spaces and special characters
3. Phone numbers need cleaning removing letters and special characters and formatting 10 digit US telephone [number](**url**)

In [0]:
bronze_customer = spark.sql(f"select * from {CATALOG}.{SCHEMA_BRONZE}.customer")

In [0]:
bronze_customer.groupBy("segment").count().display()
bronze_customer.groupBy("country").count().display()
bronze_customer.groupBy("state").count().display()
bronze_customer.groupBy("region").count().display()
bronze_customer.groupBy("postal_code").count().display()

In [0]:
bronze_customer = spark.sql(f"select * from {CATALOG}.{SCHEMA_BRONZE}.customer")
#display(bronze_customer)
#bronze_customer.groupBy("customer_id").count().where("count>1").display()
#bronze_customer.groupBy("Segment").count().where("count>1").display()

#no duplicates
#customer name needs cleaning, replacing multiple spaces and special characters
#Phone numbers need cleaning removing letters and special characters
def clean_cust_name(column):
    c = F.regexp_replace(F.col(column), r"[^a-zA-Z' -]", "")
    return F.trim(F.regexp_replace(c, r"\s+", " "))    # collapse doubles, trim ends



def clean_phone(c):
    s = F.regexp_replace(c, r"^001", "")              # 1. strip leading 001
    s = F.split(s, "x").getItem(0)                     # 2. keep part before x (extension)
    d = F.regexp_replace(s, r"\D", "")                 # 3. digits only
    return F.when(F.length(d) == 10, d)                # 4. valid → 10 digits, else null


silver_customer = (bronze_customer.withColumn("customer_name",clean_cust_name("customer_name"))
               .withColumn("phone",clean_phone("phone"))
               
               )

silver_customer.write.mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA_SILVER}.customer")
silver_customer = spark.sql(f"select * from {CATALOG}.{SCHEMA_SILVER}.customer")
silver_customer.display()


### 3.2 Exploring Product Table
Key findings:
1. product_id is duplicated
2. category and sub_category values can be extractd from product_id itself: eg: FUR-CH stands for "Furniture", "Chairs"
3. price_per_product has missing values
4. A classification look up table is planned to be created for category and subcategory mappings based on the shorthands in product_id

In [0]:
bronze_product = spark.sql(f"select * from {CATALOG}.{SCHEMA_BRONZE}.products")
display(bronze_product)

In [0]:
bronze_product.groupBy("product_id").count().where("count>1").display()
bronze_product.groupBy("category").count().display()
bronze_product.groupBy("sub_category").count().display()
bronze_product.groupBy("state").count().display()
bronze_product.groupBy("product_name").count().display()
bronze_product.groupBy("price_per_product").count().display()

#duplicate product_id exists

In [0]:
# same product_id can exist across states
bronze_product.where("product_id in ('FUR-CH-10002961','FUR-FU-10004270','OFF-PA-10000659')").orderBy("product_id").display()

In [0]:
bronze_product.groupBy("product_id","state").count().where("count>1").display()

In [0]:
# same product_id can exist across states
bronze_product.where("product_id in ('OFF-BI-10002026','TEC-PH-10002200')").orderBy("product_id").display()

In [0]:
bronze_product = spark.sql(f"select * from {CATALOG}.{SCHEMA_BRONZE}.products")
silver_product= (bronze_product.withColumn("price_per_product",(F.col("price_per_product").try_cast("Decimal(10,2)")))
                        .withColumn("product_row_id",F.monotonically_increasing_id())
            )

parts = F.split(F.col("product_id"), "-")

df_product_classification_lookup = (silver_product
    .select( "category", "sub_category","product_id")
    .withColumn("product_category_short",     parts.getItem(0))   
    .withColumn("product_sub_category_short", parts.getItem(1))
    .select(F.col("category").alias("product_category"), F.col("sub_category").alias("product_sub_category"),"product_category_short","product_sub_category_short")
    .distinct())

display(df_product_classification_lookup)


silver_product.write.mode("overwrite").option("overwriteSchema", True).saveAsTable(f"{CATALOG}.{SCHEMA_SILVER}.products")
df_product_classification_lookup.write.mode("overwrite").option("overwriteSchema", True).saveAsTable(f"{CATALOG}.{SCHEMA_SILVER}.product_classification_lookup")

### 3.3 Exploring Orders Table
Key findings
1. All customer_ids have a match in customer table
2. Not all product_ids have a match in product table
3. A product classification lookup table can be used to get the product category and sub-category
4. Order_year is calculated using order_date and is further used for the aggregations

In [0]:
bronze_orders = spark.sql(f"select * from {CATALOG}.{SCHEMA_BRONZE}.orders")
bronze_orders.where("product_id in ('OFF-BI-10002026','TEC-PH-10002200')").orderBy("product_id").display()
# price_per_prodcut can be used to map product_ids when states does not work 

In [0]:
bronze_orders = spark.sql(f"select * from {CATALOG}.{SCHEMA_BRONZE}.orders")
parts = F.split(F.col("product_id"), "-")
silver_orders = (bronze_orders.withColumn("discount",F.col("discount").try_cast("double"))
                     .withColumn("price",F.col("price").try_cast("Decimal(10,2)"))
                     .withColumn("quantity",F.col("quantity").try_cast("int"))
                    .withColumn("profit",F.col("profit").try_cast("double"))
                    .withColumn("order_date",F.to_date(F.col("order_date"),"d/M/yyyy"))
                    .withColumn("order_year",F.year(F.col("order_date")))
                    .withColumn("ship_date",F.to_date(F.col("ship_date"),"d/M/yyyy"))
                    .withColumn("price_per_product",F.col("price")/F.col("quantity").try_cast("Decimal(10,2)"))
                    .withColumn("product_category_short",     parts.getItem(0))   
                    .withColumn("product_sub_category_short", parts.getItem(1))


            )
silver_orders.write.mode("overwrite").option("overwriteSchema", True).saveAsTable(f"{CATALOG}.{SCHEMA_SILVER}.orders")

In [0]:
silver_orders = spark.sql(f"select * from {CATALOG}.{SCHEMA_SILVER}.orders")
display(silver_orders)

# END OF NOTEBOOK